In [2]:
# Standard Library
import random
import time

# Data Processing
import numpy as np
import pandas as pd
from numpy import inf

# Visualization
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from mpl_toolkits import mplot3d

# Machine Learning
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import f1_score, accuracy_score
from sklearn.svm import SVC

# Dimensionality Reduction & Utilities
import umap
from joblib import Parallel, delayed

# Feature Selection
from testflows.combinatorics import Covering
from Py_FS.wrapper.population_based.BBA import BBA as BBA


Load data set

In [3]:
df_cis = pd.read_csv('../data/cis_data.csv')
df_cacao = pd.read_csv(r'../data/data_temp/cacao.csv')
df_algarrobo = pd.read_csv(r'../data/data_temp/algarrobo.csv')
df_fruits_pures = pd.read_csv(r'../data/data_temp/MIR_Fruit_purees.csv')
df_fresh_meat = pd.read_csv(r'../data/data_temp/Fresh_meats.csv')
df_olive = pd.read_csv(r'../data/data_temp/Olive_Oils_Quadrum.csv')


df_x_cacao = df_cacao.iloc[:, 1:]
y_cacao = df_cacao.iloc[:, 0:1]
X_cacao = (df_x_cacao-df_x_cacao.min())/(df_x_cacao.max()-df_x_cacao.min())

unique_names_algarrobo = df_algarrobo['Labels'].unique()
algarrobo_x = df_algarrobo.loc[:, 'R':'REDVI']
y_algarrobo = df_algarrobo['Labels'].replace(to_replace=['N', 'P'], value=[0, 1])
X_algarrobo = (algarrobo_x-algarrobo_x.min())/(algarrobo_x.max()-algarrobo_x.min())

cis_x = df_cis[['X', 'Y', 'X10', 'Y10', 'X20', 'Y20', 'X30', 'Y30', 'X40', 'Y40']]
y_cis = df_cis[['Result']]

unique_names_berry = df_fruits_pures["label"].unique()
fruits_pures_x = df_fruits_pures.iloc[:,1:]
y_fruit_puree = df_fruits_pures.iloc[:,0:1].replace(to_replace=unique_names_berry, value =range(0,len(unique_names_berry)))
X_fruit_puree = (fruits_pures_x-fruits_pures_x.min())/(fruits_pures_x.max()-fruits_pures_x.min())


unique_names_meat = df_fresh_meat["meat"].unique()
meat_x = df_fresh_meat.iloc[:,4:]
y_meat = df_fresh_meat.iloc[:,0:1].replace(to_replace=unique_names_meat,value=range(0,len(unique_names_meat)))
X_meat =  (meat_x-meat_x.min())/(meat_x.max()-meat_x.min())

unique_names_olive = df_olive["Provenance"].unique()
olive_x = df_olive.iloc[:,3:]
y_olive = df_olive.iloc[:,2:3].replace(to_replace=unique_names_olive,value=range(0,len(unique_names_olive)))
X_olive =  (olive_x-olive_x.min())/(olive_x.max()-olive_x.min())

X_cis = (cis_x-cis_x.min())/(cis_x.max()-cis_x.min())

/var/folders/7w/w1_t98cs0k13zhwgxq2zvwj40000gn/T/ipykernel_4492/522519021.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_algarrobo = df_algarrobo['Labels'].replace(to_replace=['N', 'P'], value=[0, 1])
/var/folders/7w/w1_t98cs0k13zhwgxq2zvwj40000gn/T/ipykernel_4492/522519021.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_fruit_puree = df_fruits_pures.iloc[:,0:1].replace(to_replace=unique_names_berry, value =range(0,len(unique_names_berry)))
/var/folders/7w/w1_t98cs0k13zhwgxq2zvwj40000gn/T/ipykernel_4492/522519021.py

In [11]:
def ICAFS(dataset_X, dataset_Y, strenght,max_iteartion,model, print_logs=False):
  
  max_it = 1 

  v_variable = [0, 1]
  best_f1_score = float('-inf')
  max_iteartion_aux =  max_iteartion
  best_data_set = dataset_X.columns.values.copy()

  global_best_score = float('-inf')
  global_best_features = dataset_X.columns.values.copy()

  while max_iteartion_aux > 0:
      
      partial_score = 0
      dict_parameters = {}
      lst_of_featutres_to_check = []

      for colum_key in best_data_set:
          dict_parameters[colum_key] = v_variable
      generate_covering_array = Covering(dict_parameters, strength=strenght)
      for i,test in  enumerate(generate_covering_array.array):
          list_attributes_to_consider = []

          check_for_all_cero = True
          for (test_key, test_value) in test.items():
              if test_value == 1:
                  check_for_all_cero = False
                  list_attributes_to_consider.append(test_key)

          if check_for_all_cero:
              continue  
          lst_of_featutres_to_check.append((list_attributes_to_consider,i))
      
      clf_new = train_model(model)
      results = Parallel(n_jobs=-1)(delayed(run_cv)(clf_new, dataset_X, subset_features, dataset_Y.values.ravel(), i) for subset_features,i in lst_of_featutres_to_check)   
      sorted_results = sorted(results, key=lambda x: x[2])
      
      for score, std, i, subset_features in sorted_results:
        if score >= partial_score:
              partial_score = score
              best_std = std
              best_data_set = subset_features.copy()
      
      if partial_score > global_best_score:
            global_best_score = partial_score
            global_best_features = best_data_set.copy()

      clf_new = None
      best_f1_score = partial_score
      
      if print_logs:
        print(f"best f1 score= {best_f1_score}, iteration:{max_it}, numbers features selected ={ len(best_data_set)},best features selected={', '.join(best_data_set)}" )

      max_it = max_it +1
      max_iteartion_aux = max_iteartion_aux-1
  return global_best_features

def train_model(model_name):  

    m = eval(model_name)
    return m

def run_cv(model, X, subset_features, y, index):

    scores = cross_val_score(model, X[subset_features].values, y, cv=5, scoring='f1_macro',n_jobs=1)
    return (scores.mean(), scores.std(), index, subset_features)

In [5]:
def  plot_results_for_covering_array(scores,feature,num_of_iterarion, path_to_save_image):
        color = 'tab:blue'
        res_scores = np.array(scores)
        res_features = np.array(feature)
        res_iter = np.array(num_of_iterarion)

        plt.figure(figsize=(11, 10))
        
        fig, ax1 = plt.subplots()
        barwidth = 0.4
        color = 'tab:red'
        ax1.set_xlabel('Iterations')
        ax1.set_ylabel('Number of features', color=color)
        #ax1.set_title("ICAFS Feature selection on the Cacao dataset")
        ax1.spines['top'].set_visible(False)
        ax1.bar(res_iter-0.2, res_features, color=color, width=barwidth)
        ax1.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax1.set_ylim(1,max(res_features)+3)
        ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
        
        for bar in ax1.patches:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height}', fontsize=10,
                    ha='center', va='bottom', rotation=90)
            
        ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
        color = 'tab:blue'
        ax2.set_ylabel('F1_score', color=color)
        ax2.bar(res_iter+0.2, res_scores, color=color, width=barwidth)
        ax2.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax2.set_ylim(min(res_scores)-0.001, max(res_scores)+0.001)

        fig.tight_layout()  # otherwise the right y-label is slightly clipped
       
        for bar in ax2.patches:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height:.2f}', fontsize=10,
                    ha='center', va='bottom', rotation=90)

        #plt.title('ICAFS Feature selection for Cacao Dataset with OPF', y=-0.20)
        plt.gca().set_frame_on(False)
        plt.savefig(path_to_save_image)

In [6]:
def evaluate_dataset(X, y, dataset_name):
    
    print(f"\n--- Starting Evaluation for Dataset: {dataset_name} ---")
    
    # ---------------------------------------------------------
    # THE VAULT (Reviewer 3 Fix: Prevent Data Leakage)
    # ---------------------------------------------------------
    # Split 80% for Search/Tuning, 20% strictly for Final Testing
   
    complete_list_of_features = X.columns.values.copy()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    
    # ---------------------------------------------------------
    # PHASE 1: FEATURE SELECTION (The Scout)
    # ---------------------------------------------------------
    print("Phase 1: Running Feature Selection (kNN Evaluator)...")

    icafs_features,best_icafs_score = ICAFS(X_train,y_train,2,15,'KNeighborsClassifier()', print_logs=False)
    bba = BBA(num_agents=60, max_iter=20, train_data=X_train, train_label=y_train,save_conv_graph=False,default_mode=True,verbose=False).run()
    
    bba_features = [complete_list_of_features[i] for i in range(len(bba.Leader_agent)) if bba.Leader_agent[i] == 1]
    
    feature_dict = {
        'ICAFS': icafs_features,
        'BBA': bba_features
    }

    # ---------------------------------------------------------
    # PHASE 2 & 3: TUNING AND FINAL EVALUATION (The Polish & The Judge)
    # ---------------------------------------------------------
    print("Phase 2 & 3: Tuning SVMs and Evaluating on Vault Data...")
    
    final_scores = {}
    
    # The hyperparameter grid for the final SVM
    svm_param_grid = {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.01, 0.1],
        'kernel': ['rbf', 'linear']
    }
    
    for algo_name, selected_features in feature_dict.items():
        if len(selected_features) == 0:
            print(f"Warning: {algo_name} selected 0 features.")
            final_scores[algo_name] = 0.0
            continue
            
        # Mask the datasets to only include the selected wavelengths
        X_train_masked = X_train[selected_features]
        X_test_masked = X_test[selected_features]
        
        # Phase 2: Grid Search strictly on the 80% Training Data
        grid_search = GridSearchCV(SVC(random_state=42), svm_param_grid, cv=2, scoring='f1_macro', n_jobs=-1)
        grid_search.fit(X_train_masked, y_train.values.ravel())
        
        best_svm = grid_search.best_estimator_
        
        # Phase 3: Final Independent Evaluation on the 20% Vault Data
        y_pred = best_svm.predict(X_test_masked)
        final_f1 = f1_score(y_test, y_pred, average='macro')
        
        final_scores[algo_name] = final_f1
        print(f"  > {algo_name} Final F1-Score: {final_f1:.4f} (Features: {len(selected_features)})")
        
    return final_scores

In [7]:
#evaluate_dataset(X_cacao, y_cacao, "Cacao Dataset")

In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from scipy.stats import wilcoxon

import warnings
warnings.filterwarnings('ignore')

def evaluate_dataset_nested(X, y, dataset_name, outer_splits=5):
    
    print(f"\n{'='*50}")
    print(f"Starting NESTED CV for Dataset: {dataset_name}")
    print(f"{'='*50}")


    # THE OUTER LOOP (The Judge)
    # This splits the data into 'outer_splits' (e.g., 5) folds.
    
    complete_list_of_features = X.columns.values.copy()
    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=42)
    
    # Store the scores for every fold
    fold_scores = {'ICAFS': [], 'BBA': []}
    
    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        print(f"\n--- Processing Outer Fold {fold_idx + 1} / {outer_splits} ---")
        
        # 1. Create the Training Data and The Vault for this specific fold
        X_train, X_test = X.iloc[train_idx, :], X.iloc[test_idx, :]
        y_train, y_test = y.iloc[train_idx, :], y.iloc[test_idx, :]
        
        # ---------------------------------------------------------
        # PHASE 1: FEATURE SELECTION (The Scout)
        # ---------------------------------------------------------
        print("  > Running Feature Selection on Training Data...")
        
        
        bba = BBA(num_agents=60, max_iter=20, train_data=X_train, train_label=y_train,save_conv_graph=False,default_mode=True,verbose=False).run()
        
        icafs_features = ICAFS(X_train,y_train,2,15,'KNeighborsClassifier()', print_logs=False)
        bba_features = [complete_list_of_features[i] for i in range(len(bba.Leader_agent)) if bba.Leader_agent[i] == 1]

        feature_dict = {
            'ICAFS': icafs_features,
            'BBA': bba_features,
        }
        
        # ---------------------------------------------------------
        # PHASE 2 & 3: INNER CV TUNING AND VAULT EVALUATION
        # ---------------------------------------------------------
        print("  > Tuning SVMs and Evaluating on Vault Data...")
        
        svm_param_grid = {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.01, 0.1],
            'kernel': ['rbf', 'linear']
        }
        
        for algo_name, selected_features in feature_dict.items():
            if len(selected_features) == 0:
                fold_scores[algo_name].append(0.0)
                continue
                
            X_train_masked = X_train[selected_features]
            X_test_masked = X_test[selected_features]
            
            # Phase 2: INNER CV (GridSearchCV automatically splits X_train again internally)
            grid_search = GridSearchCV(SVC(random_state=42), svm_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
            grid_search.fit(X_train_masked, y_train)
            
            best_svm = grid_search.best_estimator_
            
            # Phase 3: Evaluate on the unseen Test Data (The Vault) for this fold
            y_pred = best_svm.predict(X_test_masked)
            fold_f1 = f1_score(y_test, y_pred, average='macro')
            
            fold_scores[algo_name].append(fold_f1)
            print(f"    - {algo_name} Fold {fold_idx + 1} F1: {fold_f1:.4f}")
            print(f"  > {algo_name} Final F1-Score: {fold_f1:.4f} (Features: {len(selected_features)})")


    # Calculate the average score across all folds
    final_avg_scores = {algo: np.mean(scores) for algo, scores in fold_scores.items()}
    
    print(f"\n>>> FINAL AVERAGED RESULTS FOR {dataset_name} <<<")
    for algo, score in final_avg_scores.items():
        print(f"  {algo}: {score:.4f}")
        
    return final_avg_scores

In [24]:
evaluate_dataset_nested(X_cacao, y_cacao, "Cacao Dataset")


Starting NESTED CV for Dataset: Cacao Dataset

--- Processing Outer Fold 1 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 1 F1: 1.0000
  > ICAFS Final F1-Score: 1.0000 (Features: 7)
    - BBA Fold 1 F1: 1.0000
  > BBA Final F1-Score: 1.0000 (Features: 413)

--- Processing Outer Fold 2 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 2 F1: 1.0000
  > ICAFS Final F1-Score: 1.0000 (Features: 5)
    - BBA Fold 2 F1: 1.0000
  > BBA Final F1-Score: 1.0000 (Features: 415)

--- Processing Outer Fold 3 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 3 F1: 1.0000
  > ICAFS Final F1-Score: 1.0000 (Features: 7)
    - BBA Fold 3 F1: 1.0000
  > BBA Final F1-Score: 1.0000 (Features: 427)

--- Processing Outer Fold 4 / 5 ---
  > Running Feature Selection on Training Data...
  > T

{'ICAFS': np.float64(0.9961317596864235),
 'BBA': np.float64(0.9990909649284442)}

In [22]:
evaluate_dataset_nested(X_fruit_puree, y_fruit_puree, "Fruit Puree Dataset")


Starting NESTED CV for Dataset: Fruit Puree Dataset

--- Processing Outer Fold 1 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 1 F1: 0.9565
  > ICAFS Final F1-Score: 0.9565 (Features: 71)
    - BBA Fold 1 F1: 0.9726
  > BBA Final F1-Score: 0.9726 (Features: 93)

--- Processing Outer Fold 2 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 2 F1: 0.9724
  > ICAFS Final F1-Score: 0.9724 (Features: 15)
    - BBA Fold 2 F1: 0.9833
  > BBA Final F1-Score: 0.9833 (Features: 62)

--- Processing Outer Fold 3 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 3 F1: 0.9389
  > ICAFS Final F1-Score: 0.9389 (Features: 10)
    - BBA Fold 3 F1: 0.9724
  > BBA Final F1-Score: 0.9724 (Features: 70)

--- Processing Outer Fold 4 / 5 ---
  > Running Feature Selection on Training Data...

{'ICAFS': np.float64(0.9613401652329312), 'BBA': np.float64(0.971213728190009)}

In [25]:
evaluate_dataset_nested(X_olive, y_olive, "Olive Dataset")


Starting NESTED CV for Dataset: Olive Dataset

--- Processing Outer Fold 1 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 1 F1: 0.8464
  > ICAFS Final F1-Score: 0.8464 (Features: 8)
    - BBA Fold 1 F1: 0.8438
  > BBA Final F1-Score: 0.8438 (Features: 177)

--- Processing Outer Fold 2 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 2 F1: 0.9702
  > ICAFS Final F1-Score: 0.9702 (Features: 59)
    - BBA Fold 2 F1: 0.9702
  > BBA Final F1-Score: 0.9702 (Features: 180)

--- Processing Outer Fold 3 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 3 F1: 0.8605
  > ICAFS Final F1-Score: 0.8605 (Features: 46)
    - BBA Fold 3 F1: 0.9080
  > BBA Final F1-Score: 0.9080 (Features: 203)

--- Processing Outer Fold 4 / 5 ---
  > Running Feature Selection on Training Data...
  >

{'ICAFS': np.float64(0.9067678812415656),
 'BBA': np.float64(0.9443934793276899)}

In [26]:
evaluate_dataset_nested(X_meat, y_meat, "Meat Dataset")


Starting NESTED CV for Dataset: Meat Dataset

--- Processing Outer Fold 1 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 1 F1: 1.0000
  > ICAFS Final F1-Score: 1.0000 (Features: 10)
    - BBA Fold 1 F1: 1.0000
  > BBA Final F1-Score: 1.0000 (Features: 130)

--- Processing Outer Fold 2 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 2 F1: 1.0000
  > ICAFS Final F1-Score: 1.0000 (Features: 104)
    - BBA Fold 2 F1: 0.9582
  > BBA Final F1-Score: 0.9582 (Features: 128)

--- Processing Outer Fold 3 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 3 F1: 0.9582
  > ICAFS Final F1-Score: 0.9582 (Features: 88)
    - BBA Fold 3 F1: 0.9582
  > BBA Final F1-Score: 0.9582 (Features: 178)

--- Processing Outer Fold 4 / 5 ---
  > Running Feature Selection on Training Data...
  

{'ICAFS': np.float64(0.9749019607843138),
 'BBA': np.float64(0.9749019607843138)}

In [27]:
evaluate_dataset_nested(X_cis, y_cis, "Cis Dataset")


Starting NESTED CV for Dataset: Cis Dataset

--- Processing Outer Fold 1 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 1 F1: 0.9323
  > ICAFS Final F1-Score: 0.9323 (Features: 2)
    - BBA Fold 1 F1: 0.9323
  > BBA Final F1-Score: 0.9323 (Features: 2)

--- Processing Outer Fold 2 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 2 F1: 0.9384
  > ICAFS Final F1-Score: 0.9384 (Features: 2)
    - BBA Fold 2 F1: 0.9505
  > BBA Final F1-Score: 0.9505 (Features: 2)

--- Processing Outer Fold 3 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 3 F1: 0.9202
  > ICAFS Final F1-Score: 0.9202 (Features: 2)
    - BBA Fold 3 F1: 0.9343
  > BBA Final F1-Score: 0.9343 (Features: 2)

--- Processing Outer Fold 4 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SV

{'ICAFS': np.float64(0.92990187672025), 'BBA': np.float64(0.9468886411186507)}